In [1]:
from langchain_openai import ChatOpenAI # pip install langchain-openai
model = ChatOpenAI(model="gpt-4o-mini")

from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 로마의휴일에 나오는 앤 공주야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 조 브레들리입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요??"),
]

model.invoke(messages)

AIMessage(content='안녕하세요, 조! 당신과 저녁을 함께 할 수 있다면 정말 좋을 것 같아요. 하지만 당신이 어쩌면 제 신분에 대해 알고 계실지도 모르겠네요. 과연 제가 당신과 저녁을 먹는 것이 가능할까요? 그래도 즐거운 시간을 보내고 싶어요! 어디에서 만날까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 66, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7771298d9', 'id': 'chatcmpl-EdUg2BsH6tgCkGRNeR4JukowLy0oR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17bfb-30ff-73ca-80d3-bab5b3cdad9d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 74, 'total_tokens': 140, 'input_tok

In [2]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()       # Create a parser instance

result = model.invoke(messages)  # Get the model's output
parser.invoke(result)            # Parse the output to a string  

'안녕하세요, 조! 당신과 저녁을 함께하는 것은 정말 기쁜 일이겠어요. 하지만 오늘 저녁은 특별한 일정이 있어서 아쉽네요. 대신, 다른 날에 꼭 같이 식사해요! 로마의 아름다운 밤을 함께 즐겨보면 좋겠네요. 어떠세요?'

In [3]:
chain = model | parser
chain.invoke(messages)

'안녕하세요, 조! 정말 기쁜 소식이네요. 저녁 식사라니, 아주 매력적인 제안이에요. 하지만 오늘은 제가 해야 할 일이 많아서요. 대신 다음에 특별한 시간을 만들어 볼까요? 그런 기회가 많이 생길 거라고 생각해요!'

In [4]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_1} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_2}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "로마의 휴일",
    "character_1": "앤 공주",
    "character_2": "조 브레들리",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 로마의 휴일에 나오는 앤 공주 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 조 브레들리입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [5]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "로마의 휴일",
    "character_1": "앤 공주",
    "character_2": "조 브레들리",
    "activity": "저녁"
})

'안녕하세요, 조! 정말 기쁜 소식이네요. 저녁 초대는 너무 감사해요. 하지만 저는 지금 이렇게 자유롭게 시간을 보내고 있답니다. 세상 밖을 탐험하며 많은 것을 배우고 싶어요. 좀 더 특별한 곳으로 함께 가볼까요?'

In [6]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "로마의 휴일",
    "character_1": "앤 공주",
    "character_2": "조 브레들리",
    "activity": "저녁"
})

Adlib(answer='안녕하세요, 조! 저녁을 함께 한다니 정말 기쁜 소식이에요. 여러분과의 저녁은 언제나 기대돼요. 어떤 곳에 가고 싶으신가요?', main_emotion='기쁨', main_emotion_intensity=0.9)